# Deliberation allocation: method in practice

**Level:** 2  
**Prerequisites:** Deliberation allocation level 1  
**Reading time:** 25 minutes

## Objectives

The learning objective is to explain the problem and its data, then inspect and, at higher levels, execute the released method on its matched task. A slow planner can return a useful proposal, a late response, invalid output or no answer. A controller must account for the planner's own outcome and latency while the environment changes; uncertainty alone does not prove that delegation helps.

## Setup

The versions below are pinned in `notebooks/requirements.txt` and are also recorded in this notebook's metadata.

```bash
pip install reflexmesh==0.1.2 numpy==2.4.6 matplotlib==3.11.2 torch==2.14.1 joblib==1.6.0 gymnasium==1.4.0 scikit-learn==1.9.1 stable-baselines3==2.9.2 sb3-contrib==2.9.2
```

## Data

The notebook reads committed product artifacts. Owner: Neuraxis. Repository: https://github.com/fsantibanezleal/CAOS_Neuraxis. License: Apache-2.0. The JSON is a first-party, frozen experiment artifact; no external benchmark data are redistributed in this notebook.

This notebook reads `data/derived/index.json` and the listed trace artifacts. The method run uses an isolated Reflexmesh case workspace. For learned methods, fetch the exact public release assets with `python -m reflexmesh.cli fetch-models --destination models/checkpoints --version v0.01.000`, then set `NEURAXIS_CHECKPOINTS` to that folder. The release manifest verifies asset hashes. Planner methods M09/M10 are read from frozen traces unless `NEURAXIS_NOTEBOOK_MODEL` explicitly names a configured local Ollama model.

Artifacts for this lesson:

- `data/derived/index.json`
- `data/derived/semantic-routing-study/analysis.json`
- `data/derived/episodes/M11-C13-nominal-40000.json`
- `data/derived/episodes/M12-C13-nominal-40000.json`
- `data/derived/episodes/M09-C13-nominal-40000.json`
- `data/derived/episodes/M10-C13-nominal-40000.json`

## Method

Inspect M09 direct planning, M10 asynchronous delegation, M11 learned deferral and M12 metacontrol as separate methods. Recompute the observed routing-study counts from the persisted report and compare its held-out arms without pooling them into the core benchmark.

A value-of-computation estimate is \(VOC(c\mid x)=\mathbb E[V(x'\mid c,x)]-V_{act}(x)-\lambda_c cost(c)-\lambda_d delay(c)\). The expected value must include failed, late, invalid and superseded plans. This is the research formulation; this notebook does not refit or silently improve the historical gate.

### Check the pinned analysis environment.

In [1]:
# Setup is performed outside the notebook; report the exact runtime pins.
import importlib.metadata
for distribution in ('reflexmesh', 'numpy', 'matplotlib', 'torch', 'joblib', 'gymnasium', 'scikit-learn', 'stable-baselines3', 'sb3-contrib'):
    print('{}=={}'.format(distribution, importlib.metadata.version(distribution)))

reflexmesh==0.1.2
numpy==2.4.6
matplotlib==3.11.2
torch==2.14.1
joblib==1.6.0
gymnasium==1.4.0
scikit-learn==1.9.1
stable-baselines3==2.9.2
sb3-contrib==2.9.2


### Verify the audit and each trace's bytes before interpreting outcomes.

In [2]:
# Read committed evidence and use the released API; never define another policy.
import hashlib, json
from pathlib import Path
ROOT = Path('../../../').resolve()
index_path = ROOT / 'data/derived/index.json'
index_bytes = index_path.read_bytes()
index = json.loads(index_bytes)
audit = json.loads((ROOT / 'data/derived/benchmark-adequacy.json').read_text(encoding='utf-8'))
assert hashlib.sha256(index_bytes).hexdigest() == audit['index_sha256']
selected_methods = ['M11', 'M12', 'M09', 'M10']
sample_case = 'C13'
seed = 40000
records = {}
for method_id in selected_methods:
    row = next(r for r in index['episodes'] if r['method_id'] == method_id and r['case_id'] == sample_case and r['variant'] == 'nominal' and r['seed'] == seed)
    trace_path = ROOT / 'data/derived' / row['path']
    trace_bytes = trace_path.read_bytes()
    assert hashlib.sha256(trace_bytes).hexdigest() == row['sha256']
    records[method_id] = json.loads(trace_bytes)
print('Index digest:', hashlib.sha256(index_bytes).hexdigest())
print('Evidence records:', len(records), '| case:', sample_case, '| seed:', seed)
print('Methods:', ', '.join(selected_methods))

Index digest: 1024a451fdc1f1aebc4e8d8d66e6bf1aca119f26ae058d3de9b84ff312d8d7a6
Evidence records: 4 | case: C13 | seed: 40000
Methods: M11, M12, M09, M10


### Inspect the original decisions and outcomes

In [3]:
# Summarize the recorded first action, authorized candidates, and independent oracle outcome.
for method_id, episode in records.items():
    first = episode['decisions'][0]
    allowed = [candidate['action'] for candidate in first['candidates'] if candidate['allowed']]
    print(method_id, '| action=', first['action'], '| reason=', first['reason'], '| allowed=', allowed)
    print('  success=', episode['metrics']['task_success'], '| steps=', episode['metrics']['steps'], '| violations=', episode['metrics']['violations'])

M11 | action= a-wait | reason= learned_incremental_benefit_gate | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 1 | steps= 2 | violations= 0
M12 | action= a-wait | reason= persistent_outcome_aware_metacontrol | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 1 | steps= 2 | violations= 0
M09 | action= a-ack_event | reason= Acknowledge the received event to confirm state processing and maintain responsiveness as per the goal. | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 0 | steps= 16 | violations= 0
M10 | action= a-wait | reason= asynchronous_fsm_planner | allowed= ['a-ack_event', 'a-wait', 'a-refresh', 'a-cancel', 'a-collect', 'a-wait_dependency']
  success= 1 | steps= 2 | violations= 0


### Execute the released policies and compare the run

In [4]:
# Planner outcomes come from frozen traces; no LLM call is implicit.
import os
from pathlib import Path
from reflexmesh.environments import CaseSpec, SoftwareEnvironment
from reflexmesh.pipeline import make_policy
from reflexmesh.planners import OllamaPlanner
checkpoint_dir = Path(os.environ.get('NEURAXIS_CHECKPOINTS', str(ROOT / 'models/checkpoints')))
planner_model = os.environ.get('NEURAXIS_NOTEBOOK_MODEL')
reruns = {}
for method_id in selected_methods:
    if method_id in ('M09', 'M10') and not planner_model:
        print(method_id, 'not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above')
        continue
    planner = OllamaPlanner(planner_model) if planner_model and method_id in ('M09', 'M10', 'M11', 'M12') else None
    policy = make_policy(method_id, checkpoint_dir, planner=planner)
    policy.reset('neuraxis-notebook-' + method_id)
    with SoftwareEnvironment(CaseSpec(sample_case, 'nominal', seed)) as env:
        while not env.terminal:
            _, outcome = env.step(policy.predict(env.observe()))
        success = env.verify()
    recorded = records[method_id]
    reruns[method_id] = {'success': success, 'recorded_success': bool(recorded['metrics']['task_success']), 'recorded_first_action': recorded['decisions'][0]['action']}
    print(method_id, reruns[method_id])

M11 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-wait'}
M12 {'success': True, 'recorded_success': True, 'recorded_first_action': 'a-wait'}
M09 not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above
M10 not rerun: planner needs explicit NEURAXIS_NOTEBOOK_MODEL; frozen result remains above


### Compare the full sequence, not only the first action

In [5]:
# Sequence equality is diagnostic; the independent verifier determines task success.
for method_id, episode in records.items():
    actions = [decision['action'] for decision in episode['decisions']]
    print(method_id, ' -> '.join(actions), '| terminal=', episode['status'], '| verified=', episode['metrics']['task_success'])

M11 a-wait -> a-collect | terminal= succeeded | verified= 1
M12 a-wait -> a-collect | terminal= succeeded | verified= 1
M09 a-ack_event -> a-wait -> a-ack_event -> a-ack_event -> a-wait_dependency -> a-ack_event -> a-ack_event -> a-refresh -> a-refresh -> a-wait -> a-ack_event -> a-wait -> a-ack_event -> a-ack_event -> a-ack_event -> a-ack_event | terminal= failed | verified= 0
M10 a-wait -> a-collect | terminal= succeeded | verified= 1


## Results

The gate fit used 228 states with seven positive labels and selected threshold 1.01, meaning it never delegated through that gate. M12's other trigger is separate. Report planner calls, latency and verified success; a configured LLM is not evidence of useful deliberation.

No generated outputs are committed by this authoring step; `scripts/run_notebooks.py` executes and stores them from a fresh kernel.

## Summary

Delegation must earn its cost from paired outcomes. The current evidence reports a negative learned-gate result and no demonstrated task-success advantage for M12.

## References

- [JEV](https://typesafe.ai/blog/introducing-system-one-models-and-jev)
- [DPT](https://aclanthology.org/2025.acl-long.206/)
- [VOC](https://arxiv.org/abs/1711.06892)
- [DEFER](https://proceedings.mlr.press/v119/mozannar20b.html)